# Hardware Log GIF Generator — 3-Panel GPR View

3-panel animated GIFs for yes_wind_estimation hardware logs:
- **Left**: trajectory + reachable tubes + two wind arrows from drone
- **Middle**: TVGPR for wy = f(time, z) — wind in Y direction vs height
- **Right**: TVGPR for wz = f(time, y) — wind in Z direction vs horizontal position

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import glob
import os
import sys
import jax.numpy as jnp

sys.path.insert(0, '/home/egmc/ws_px4_rta_mm_gpr/src/px4_rta_mm_gpr/px4_rta_mm_gpr/jax_mm_rta')
from TVGPR import TVGPR

plt.rcParams.update({
    'text.usetex': False,
    'font.family': 'sans-serif',
    'font.size': 14
})

print('Imports complete.')

Imports complete.


In [2]:
# ====== CONFIGURATION ======
HORIZON_STEPS = 12
FRAME_SKIP    = 3
GIF_FPS       = 10
FONTSIZE      = 18
LINEWIDTH     = 1.5
QUADSIZE      = 0.25
OBS_WINDOW    = 50
WIND_Y_SCALE  = 15
WIND_Z_SCALE  = 20

GPR_N_POINTS  = 60    # spatial query points for GP curves
GPR_N_SIGMA   = 3     # confidence bound multiplier
OBSSIZE       = 100   # scatter dot size

SIGMA_F  = 5.0
LENGTH   = 2.0
SIGMA_N  = 0.01
EPSILON  = 0.25

# Resolve BASE_DIR whether notebook is run from data_analysis/ or from its own directory
_rel = 'log_files/hardware'
BASE_DIR = _rel if os.path.isdir(os.path.join(_rel, 'yes_wind_estimation')) else '.'

In [3]:
def quadplot(ypos, zpos, rot, scale):
    ts = -0.5 * np.arange(np.pi, 3/2 * np.pi, 0.2) + 0.3
    xtemp = np.cos(ts)
    ytemp = np.sin(ts) * np.cos(ts)
    xs = np.hstack((0.4*xtemp - 1, -1, -1, 1, 1, 0.4*xtemp + 1))
    ys = np.hstack((0.3*ytemp + 0.4, 0.4, 0, 0, 0.4, 0.3*ytemp + 0.4))
    xs, ys = scale * xs, scale * ys
    rotmat = np.array([[np.cos(rot), -np.sin(rot)],
                       [np.sin(rot),  np.cos(rot)]])
    newpos = rotmat @ np.vstack((xs, ys))
    return newpos[0, :] + ypos, newpos[1, :] + zpos


def get_RMSE(data, num_refs_per_step=12):
    y     = data['y'].to_numpy()
    z     = data['z'].to_numpy()
    y_ref = data['y_ref'].to_numpy()
    z_ref = data['z_ref'].to_numpy()
    n_true, n_ref = len(y), len(y_ref)
    squared_errors = []
    n = 0
    while n < n_true:
        if np.isnan(y[n]) or np.isnan(z[n]):
            break
        ref_idx = num_refs_per_step * n
        if ref_idx >= n_ref:
            break
        if np.isnan(y_ref[ref_idx]) or np.isnan(z_ref[ref_idx]):
            n += 1
            continue
        squared_errors.append((y[n] - y_ref[ref_idx])**2 + (z[n] - z_ref[ref_idx])**2)
        n += 1
    return np.sqrt(np.mean(squared_errors)) if squared_errors else np.nan


def generate_gif_gpr(log_path, gif_path):
    """
    3-panel animated GIF from a hardware log file.

    Left panel:   trajectory + reachable tubes + two wind arrows from drone
    Middle panel: TVGPR for wy = f(time, z) plotted over z (height)
    Right panel:  TVGPR for wz = f(time, y) plotted over y (horizontal)
    """
    data = pd.read_csv(log_path)
    if len(data) < HORIZON_STEPS * 2:
        print(f'  SKIP (too few rows: {len(data)})')
        return None

    num_horizon  = HORIZON_STEPS
    max_timestep = len(data) // num_horizon

    time_all  = data['time'].values
    y_all     = data['y'].values
    z_all     = -data['z'].values
    yaw_all   = data['yaw'].values
    wy_all    = data['wy'].values
    wz_all    = data['wz'].values
    y_ref_all = data['y_ref'].values
    z_ref_all = -data['z_ref'].values
    pyH_all   = data['save_tube_pyH'].values
    pyL_all   = data['save_tube_pyL'].values
    pzH_all   = -data['save_tube_pzL'].values
    pzL_all   = -data['save_tube_pzH'].values

    has_wind = (np.nanmax(np.abs(wy_all)) > 1e-6) or (np.nanmax(np.abs(wz_all)) > 1e-6)
    rmse = get_RMSE(data, num_refs_per_step=num_horizon)

    # TVGPR init
    tvgp_wy = tvgp_wz = None
    if has_wind:
        n_init = min(20, max_timestep)
        idx    = np.arange(n_init)
        vm_wy  = ~(np.isnan(time_all[idx]) | np.isnan(z_all[idx]) | np.isnan(wy_all[idx]))
        vm_wz  = ~(np.isnan(time_all[idx]) | np.isnan(y_all[idx]) | np.isnan(wz_all[idx]))
        obs_wy = np.column_stack((time_all[idx][vm_wy], z_all[idx][vm_wy], wy_all[idx][vm_wy]))
        obs_wz = np.column_stack((time_all[idx][vm_wz], y_all[idx][vm_wz], wz_all[idx][vm_wz]))
        tvgp_wy = TVGPR(jnp.array(obs_wy), sigma_f=SIGMA_F, l=LENGTH, sigma_n=SIGMA_N, epsilon=EPSILON)
        tvgp_wz = TVGPR(jnp.array(obs_wz), sigma_f=SIGMA_F, l=LENGTH, sigma_n=SIGMA_N, epsilon=EPSILON)

    # Spatial limits
    y_valid = y_all[~np.isnan(y_all)]
    z_valid = z_all[~np.isnan(z_all)]
    ym = (np.max(y_valid) - np.min(y_valid)) * 0.15
    zm = (np.max(z_valid) - np.min(z_valid)) * 0.15
    y_min, y_max = np.min(y_valid) - ym, np.max(y_valid) + ym
    z_min, z_max = np.min(z_valid) - zm, np.max(z_valid) + zm

    # Wind magnitude limits for GPR panels
    wy_abs = np.nanmax(np.abs(wy_all)) if has_wind else 1.0
    wz_abs = np.nanmax(np.abs(wz_all)) if has_wind else 1.0
    w_max  = max(wy_abs, wz_abs, 0.5) * 1.4

    # GPR spatial query grids
    gpr_z_grid = np.linspace(z_min, z_max, GPR_N_POINTS)  # wy = f(t, z)
    gpr_y_grid = np.linspace(y_min, y_max, GPR_N_POINTS)  # wz = f(t, y)

    # ----------------------------------------------------------------
    # Figure
    # ----------------------------------------------------------------
    fig, (ax, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 7))

    # === LEFT PANEL ===
    wind_arrow_y = wind_arrow_z = None
    if has_wind:
        # Fully opaque, thick arrows rendered above everything else
        wind_arrow_y = ax.quiver([0], [0], [0], [0], color='tab:gray',   alpha=1.0,
                                 scale=WIND_Y_SCALE, width=0.010, headwidth=5, zorder=15)
        wind_arrow_z = ax.quiver([0], [0], [0], [0], color='darkorange', alpha=1.0,
                                 scale=WIND_Z_SCALE, width=0.010, headwidth=5, zorder=15)

    actual_line, = ax.plot([], [], color='black', linewidth=LINEWIDTH, label='Path Followed')
    reference_horizon, = ax.plot([], [], linestyle='dashed', color='tab:blue',
                                  linewidth=LINEWIDTH, alpha=0.9, marker='o', markersize=4,
                                  label='Reference Trajectory')
    tube_rects = []
    for i in range(num_horizon):
        rect = Rectangle((0, 0), 1, 1, linewidth=1.5,
                          edgecolor='red', facecolor='red', alpha=0.15)
        ax.add_patch(rect)
        tube_rects.append(rect)
    quad_line,   = ax.plot([], [], color='black', linewidth=LINEWIDTH)
    current_pos, = ax.plot([], [], 'ko', markersize=8)

    ax.set_xlabel('y', fontsize=FONTSIZE)
    ax.set_ylabel('z', fontsize=FONTSIZE)
    title_obj = ax.set_title('t = 0.00', fontsize=FONTSIZE)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(y_min, y_max)
    ax.set_ylim(z_min, z_max)
    ax.legend(loc='upper right', fontsize=FONTSIZE - 4)

    # === MIDDLE PANEL: wy = f(t, z) ===
    mean_line_y,  = ax2.plot([], [], color='tab:blue', linewidth=LINEWIDTH)
    upper_conf_y, = ax2.plot([], [], color='lightblue', linewidth=LINEWIDTH)
    lower_conf_y, = ax2.plot([], [], color='lightblue', linewidth=LINEWIDTH)
    # Colors set per-frame via set_facecolor using viridis; no cmap needed at init
    obs_scatter_y = ax2.scatter([], [], s=OBSSIZE, edgecolors='black', linewidths=0.5, zorder=5)

    ax2.set_xlabel('z', fontsize=FONTSIZE)
    ax2.set_ylabel('Wind Force (w_y)', fontsize=FONTSIZE)
    ax2.set_title('Disturbance Behavior (y)', fontsize=FONTSIZE)
    ax2.set_xlim(z_min, z_max)
    ax2.set_ylim(-w_max, w_max)
    ax2.axhline(0, color='gray', linewidth=0.8, alpha=0.4)
    ax2.grid(True, alpha=0.3)
    legend_elems = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='purple',
               markersize=8, markeredgecolor='black', label='Observations'),
        Line2D([0], [0], color='tab:blue',  linewidth=LINEWIDTH, label='Mean'),
        Line2D([0], [0], color='lightblue', linewidth=LINEWIDTH, label='Confidence Bound'),
    ]
    ax2.legend(handles=legend_elems, loc='lower right', fontsize=FONTSIZE - 4)

    # === RIGHT PANEL: wz = f(t, y) ===
    mean_line_z,  = ax3.plot([], [], color='tab:blue', linewidth=LINEWIDTH)
    upper_conf_z, = ax3.plot([], [], color='lightblue', linewidth=LINEWIDTH)
    lower_conf_z, = ax3.plot([], [], color='lightblue', linewidth=LINEWIDTH)
    obs_scatter_z = ax3.scatter([], [], s=OBSSIZE, edgecolors='black', linewidths=0.5, zorder=5)

    ax3.set_xlabel('y', fontsize=FONTSIZE)
    ax3.set_ylabel('Wind Force (w_z)', fontsize=FONTSIZE)
    ax3.set_title('Disturbance Behavior (z)', fontsize=FONTSIZE)
    ax3.set_xlim(y_min, y_max)
    ax3.set_ylim(-w_max, w_max)
    ax3.axhline(0, color='gray', linewidth=0.8, alpha=0.4)
    ax3.grid(True, alpha=0.3)
    ax3.legend(handles=legend_elems, loc='lower right', fontsize=FONTSIZE - 4)

    plt.tight_layout()

    state = {'tvgp_wy': tvgp_wy, 'tvgp_wz': tvgp_wz}

    # ----------------------------------------------------------------
    # Animation update
    # ----------------------------------------------------------------
    def update(n):
        ret = [actual_line, reference_horizon, quad_line, current_pos, title_obj,
               mean_line_y, upper_conf_y, lower_conf_y, obs_scatter_y,
               mean_line_z, upper_conf_z, lower_conf_z, obs_scatter_z]
        if wind_arrow_y is not None:
            ret.append(wind_arrow_y)
        if wind_arrow_z is not None:
            ret.append(wind_arrow_z)
        ret.extend(tube_rects)

        if n >= max_timestep or np.isnan(time_all[n]):
            return ret

        actual_line.set_data(y_all[:n+1], z_all[:n+1])
        t_curr = time_all[n]
        y_curr, z_curr, yaw_curr = y_all[n], z_all[n], yaw_all[n]

        title_obj.set_text(f't = {t_curr:.2f}')

        if has_wind:
            # --- TVGPR sliding window update ---
            s   = max(0, n - OBS_WINDOW)
            idx = np.arange(s, n + 1)
            vm_wy = ~(np.isnan(time_all[idx]) | np.isnan(z_all[idx]) | np.isnan(wy_all[idx]))
            vm_wz = ~(np.isnan(time_all[idx]) | np.isnan(y_all[idx]) | np.isnan(wz_all[idx]))
            if np.sum(vm_wy) > 5:
                obs = np.column_stack((time_all[idx][vm_wy], z_all[idx][vm_wy], wy_all[idx][vm_wy]))
                state['tvgp_wy'] = TVGPR(jnp.array(obs), sigma_f=SIGMA_F, l=LENGTH,
                                         sigma_n=SIGMA_N, epsilon=EPSILON)
            if np.sum(vm_wz) > 5:
                obs = np.column_stack((time_all[idx][vm_wz], y_all[idx][vm_wz], wz_all[idx][vm_wz]))
                state['tvgp_wz'] = TVGPR(jnp.array(obs), sigma_f=SIGMA_F, l=LENGTH,
                                         sigma_n=SIGMA_N, epsilon=EPSILON)

            # --- Wind arrows from drone ---
            if wind_arrow_y is not None and state['tvgp_wy'] is not None:
                wy_val = float(state['tvgp_wy'].mean(jnp.array([t_curr, z_curr]))[0][0])
                wind_arrow_y.set_offsets([[y_curr, z_curr]])
                wind_arrow_y.set_UVC([wy_val], [0])
            if wind_arrow_z is not None and state['tvgp_wz'] is not None:
                wz_val = float(state['tvgp_wz'].mean(jnp.array([t_curr, y_curr]))[0][0])
                wind_arrow_z.set_offsets([[y_curr, z_curr]])
                wind_arrow_z.set_UVC([0], [wz_val])

            # --- Middle panel: wy GP over z ---
            if state['tvgp_wy'] is not None:
                means = np.zeros(GPR_N_POINTS)
                stds  = np.zeros(GPR_N_POINTS)
                for i, zp in enumerate(gpr_z_grid):
                    f_bar, cov = state['tvgp_wy'].fit(jnp.array([t_curr, zp]))
                    means[i] = float(f_bar[0, 0])
                    stds[i]  = float(jnp.sqrt(jnp.abs(cov[0, 0])))
                sig = GPR_N_SIGMA * stds
                mean_line_y.set_data(gpr_z_grid, means)
                upper_conf_y.set_data(gpr_z_grid, means + sig)
                lower_conf_y.set_data(gpr_z_grid, means - sig)

                obs_t  = time_all[idx][vm_wy]
                obs_zz = z_all[idx][vm_wy]
                obs_w  = wy_all[idx][vm_wy]
                if len(obs_t) > 0:
                    t_min, t_max = obs_t.min(), obs_t.max()
                    norm_t = (obs_t - t_min) / (t_max - t_min + 1e-9)
                    colors = plt.cm.viridis(norm_t)
                    obs_scatter_y.set_offsets(np.column_stack([obs_zz, obs_w]))
                    obs_scatter_y.set_facecolor(colors)
                    obs_scatter_y.set_sizes([OBSSIZE] * len(obs_t))

            # --- Right panel: wz GP over y ---
            if state['tvgp_wz'] is not None:
                means = np.zeros(GPR_N_POINTS)
                stds  = np.zeros(GPR_N_POINTS)
                for i, yp in enumerate(gpr_y_grid):
                    f_bar, cov = state['tvgp_wz'].fit(jnp.array([t_curr, yp]))
                    means[i] = float(f_bar[0, 0])
                    stds[i]  = float(jnp.sqrt(jnp.abs(cov[0, 0])))
                sig = GPR_N_SIGMA * stds
                mean_line_z.set_data(gpr_y_grid, means)
                upper_conf_z.set_data(gpr_y_grid, means + sig)
                lower_conf_z.set_data(gpr_y_grid, means - sig)

                obs_t  = time_all[idx][vm_wz]
                obs_yy = y_all[idx][vm_wz]
                obs_w  = wz_all[idx][vm_wz]
                if len(obs_t) > 0:
                    t_min, t_max = obs_t.min(), obs_t.max()
                    norm_t = (obs_t - t_min) / (t_max - t_min + 1e-9)
                    colors = plt.cm.viridis(norm_t)
                    obs_scatter_z.set_offsets(np.column_stack([obs_yy, obs_w]))
                    obs_scatter_z.set_facecolor(colors)
                    obs_scatter_z.set_sizes([OBSSIZE] * len(obs_t))

        # --- Reachable tubes ---
        rs, re = num_horizon * n, num_horizon * n + num_horizon
        if re <= len(data):
            reference_horizon.set_data(y_ref_all[rs:re], z_ref_all[rs:re])
            for i in range(num_horizon):
                pH, pL = pyH_all[rs+i], pyL_all[rs+i]
                zH, zL = pzH_all[rs+i], pzL_all[rs+i]
                if not (np.isnan(pH) or np.isnan(pL) or np.isnan(zH) or np.isnan(zL)):
                    tw, th = pH - pL, zH - zL
                    if 0 < tw < 50 and 0 < th < 50:
                        tube_rects[i].set_xy((pL, zL))
                        tube_rects[i].set_width(tw)
                        tube_rects[i].set_height(th)
                        tube_rects[i].set_visible(True)
                    else:
                        tube_rects[i].set_visible(False)
                else:
                    tube_rects[i].set_visible(False)
        else:
            reference_horizon.set_data([], [])
            for r in tube_rects:
                r.set_visible(False)

        # --- Quadrotor ---
        if not (np.isnan(y_curr) or np.isnan(z_curr)):
            qy, qz = quadplot(y_curr, z_curr, yaw_curr, QUADSIZE)
            quad_line.set_data(qy, qz)
            quad_line.set_visible(True)
            current_pos.set_data([y_curr], [z_curr])
            current_pos.set_visible(True)
        else:
            quad_line.set_visible(False)
            current_pos.set_visible(False)

        return ret

    frames = list(range(0, max_timestep, FRAME_SKIP))
    ani = animation.FuncAnimation(fig, update, frames=frames,
                                  interval=50, blit=True, repeat=False)
    ani.save(gif_path, writer='pillow', fps=GIF_FPS)
    plt.close(fig)

    print(f'  Saved: {gif_path}  ({len(frames)} frames, RMSE={rmse:.4f}, wind={has_wind})')
    return rmse

## Yes Wind Estimation — 3-Panel GPR GIFs

In [ ]:
folder = os.path.join(BASE_DIR, 'yes_wind_estimation')
log_files = sorted(glob.glob(os.path.join(folder, '*.log')))
print(f'Found {len(log_files)} log files in {folder}\n')

results = {}
for lf in log_files:
    name = os.path.splitext(os.path.basename(lf))[0]
    gif_out = os.path.join(folder, f'{name}_gpr.gif')
    print(f'Processing {os.path.basename(lf)} ...')
    rmse = generate_gif_gpr(lf, gif_out)
    if rmse is not None:
        results[name] = rmse

print('\n--- Summary ---')
for name, rmse in results.items():
    print(f'  {name}: RMSE = {rmse:.6f}')

Found 4 log files in ./yes_wind_estimation

Processing hw1_yes_wind.log ...
